# 📦 Carga de Catálogo de Productos → PostgreSQL

Este notebook lee el Excel de productos y carga los datos en la base de datos PostgreSQL.

**Puede ejecutarse tantas veces como sea necesario** — todos los inserts usan `ON CONFLICT DO NOTHING` o `DO UPDATE` para ser idempotentes.

### Pasos:
1. Instalar dependencias
2. Definir la ruta del Excel (`data/productos.xlsx`)
3. Configurar la conexión a la BD
4. Ejecutar la carga
5. Verificar resultados

---
## 1. Instalar dependencias

In [22]:
!pip install openpyxl psycopg2-binary pandas --quiet
print('✅ Dependencias instaladas')


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✅ Dependencias instaladas


---
## 2. Ruta del archivo Excel

El catálogo se lee desde `data/productos.xlsx` (relativo a la carpeta `cerpal_backend`). Ejecuta la celda siguiente para confirmar la ruta.

In [23]:
EXCEL_PATH = "data/productos.xlsx"
print(f'✅ Archivo a cargar: {EXCEL_PATH}')

✅ Archivo a cargar: data/productos.xlsx


---
## 3. Configurar conexión a PostgreSQL

Rellena los datos de tu servidor. Si usas pgAdmin local, necesitarás exponer el puerto o usar un túnel.

> **Opciones comunes:**
> - **Local con ngrok**: usa el host/puerto de ngrok
> - **Supabase / Neon / Railway**: copia la connection string del dashboard
> - **VPS propio**: IP pública + puerto 5432 abierto en firewall

In [24]:
# ─── CONFIGURA AQUÍ TUS DATOS DE CONEXIÓN ───────────────────
DB_CONFIG = {
    'host':     'localhost',        # ej: 'localhost', '34.xx.xx.xx', 'db.supabase.co'
    'port':     5432,
    'database': 'cerpal',
    'user':     'cerpal',
    'password': 'cerpal',
}
# ─────────────────────────────────────────────────────────────

import psycopg2

def get_conn():
    return psycopg2.connect(**DB_CONFIG)

# Test de conexión
try:
    conn = get_conn()
    cur = conn.cursor()
    cur.execute('SELECT version();')
    print('✅ Conexión OK:', cur.fetchone()[0])
    conn.close()
except Exception as e:
    print('❌ Error de conexión:', e)

✅ Conexión OK: PostgreSQL 16.13 on aarch64-unknown-linux-musl, compiled by gcc (Alpine 15.2.0) 15.2.0, 64-bit


---
## 4. Leer y parsear el Excel

In [25]:
from openpyxl import load_workbook

wb = load_workbook(EXCEL_PATH, read_only=True, data_only=True)
print('Hojas encontradas:', wb.sheetnames)

# ── Hoja 1: Articulos Precio ──────────────────────────────────
ws_prod = wb['Articulos Precio']
productos_raw = list(ws_prod.iter_rows(values_only=True))[1:]  # saltar cabecera

# ── Hoja 2: Atributos Valores ─────────────────────────────────
ws_avals = wb['Atributos Valores']
avals_raw = list(ws_avals.iter_rows(values_only=True))[1:]

# ── Hoja 3: Atributos ─────────────────────────────────────────
ws_attrs = wb['Atributos']
attrs_raw = list(ws_attrs.iter_rows(values_only=True))[1:]

print(f'  Productos:           {len(productos_raw):>5}')
print(f'  Filas Atrib.Valores: {len(avals_raw):>5}')
print(f'  Filas Atributos:     {len(attrs_raw):>5}')

Hojas encontradas: ['Articulos Precio', 'Atributos Valores', 'Atributos']
  Productos:            2300
  Filas Atrib.Valores:   747
  Filas Atributos:      3911


In [26]:
# ── Extraer nombres de atributos (evitando fórmulas) ─────────
attr_names = {}   # {external_id: name}
for r in attrs_raw:
    code_attr  = r[2]   # Cód. Atributo
    name_attr  = r[3]   # Nombre Atributo
    if (code_attr and name_attr
            and isinstance(name_attr, str)
            and not name_attr.startswith('=')):
        attr_names[code_attr] = name_attr

print(f'Atributos únicos detectados: {len(attr_names)}')
for k, v in list(attr_names.items())[:8]:
    print(f'  [{k}] {v}')
print('  ...')

Atributos únicos detectados: 39
  [2] Ancho Vinilo Textil
  [16] Premium
  [20] Terciopelo
  [4] Anchos Bobina de Lona
  [29] Tipo de Lona
  [9] Capacidad Cartucho
  [43] Tipo de Cartucho
  [10] Color de Tinta
  ...


In [27]:
# ── Extraer valores de atributo ───────────────────────────────
# attr_values: {attr_ext_id: [(line_id, valor, color)]}
attr_values = {}
for r in avals_raw:
    attr_code = r[0]   # Code  = id externo del atributo
    line_id   = r[2]   # LineId
    valor     = r[3]   # Valor
    color     = r[4]   # Codigo HTML del color
    if attr_code and valor:
        attr_values.setdefault(attr_code, []).append((line_id, valor, color))

total_vals = sum(len(v) for v in attr_values.values())
print(f'Grupos de atributo con valores: {len(attr_values)}')
print(f'Total valores de atributo:      {total_vals}')

Grupos de atributo con valores: 52
Total valores de atributo:      745


In [28]:
# ── Extraer líneas producto-atributo-valor ────────────────────
# prod_attr_lines: lista de dicts con la relación variante → atributo → valor
prod_attr_lines = []
for r in attrs_raw:
    prod_ref   = r[0]   # Code (referencia del producto)
    cod_attr   = r[2]   # Cód. Atributo
    nombre_attr= r[3]   # Nombre Atributo
    line_id    = r[5]   # LineId
    cod_valor  = r[6]   # Cód. Valor
    nombre_val = r[7]   # Nombre Valor

    if not (prod_ref and cod_attr and cod_valor and nombre_val):
        continue
    if isinstance(nombre_val, str) and nombre_val.startswith('='):
        continue

    attr_name_resolved = (
        nombre_attr
        if isinstance(nombre_attr, str) and not nombre_attr.startswith('=')
        else attr_names.get(cod_attr, f'Attr_{cod_attr}')
    )

    prod_attr_lines.append({
        'product_ref': prod_ref,
        'attr_id':     cod_attr,
        'attr_name':   attr_name_resolved,
        'line_id':     line_id,
        'value_id':    cod_valor,
        'value_name':  nombre_val,
    })

print(f'Líneas producto-atributo-valor: {len(prod_attr_lines)}')

Líneas producto-atributo-valor: 3894


In [29]:
# ── Construir jerarquía template / variante ───────────────────
templates = {}    # {code: (name, active, price)}
variants  = []    # [(code, name, padre_code, active, integrable, price)]

for r in productos_raw:
    cod    = r[0]
    desc   = r[1]
    activo = r[2]
    integ  = r[3]
    padre  = r[4]
    precio = r[5] or 0.0

    active_b = str(activo).lower() in ('sí', 'si', 'yes', 'true', '1')
    integ_b  = str(integ).lower()  in ('sí', 'si', 'yes', 'true', '1')

    if not padre or padre == cod:
        templates[cod] = (desc, active_b, float(precio))

    variants.append((cod, desc, padre if padre else cod, active_b, integ_b, float(precio)))

# Mapa rápido: variant_code → padre_code
variant_to_tmpl = {v[0]: v[2] for v in variants}

print(f'Templates:  {len(templates)}')
print(f'Variantes:  {len(variants)}')

Templates:  1190
Variantes:  2300


---
## 5. Funciones de carga

Cada función es **idempotente** — si el registro ya existe lo ignora o actualiza el precio.

In [39]:
import psycopg2.extras

def load_attributes(conn):
    """Carga product_attribute."""
    data = [
        (ext_id, name)
        for ext_id, name in attr_names.items()
        if isinstance(ext_id, int)
    ]
    with conn.cursor() as cur:
        psycopg2.extras.execute_batch(cur, """
            INSERT INTO product_attribute (external_id, name)
            VALUES (%s, %s)
            ON CONFLICT (external_id) DO UPDATE SET
                name       = EXCLUDED.name,
                write_date = NOW()
        """, data)
    conn.commit()
    print(f'  ✅ product_attribute:       {len(data):>5} registros')


def load_attribute_values(conn):
    data = []
    for attr_code, vals in attr_values.items():
        if not isinstance(attr_code, int):
            continue
        for (line_id, valor, color) in vals:
            if not valor:
                continue
            c = color if (color and color != '#ffffff') else None
            data.append((line_id, attr_code, valor, c, line_id))

    with conn.cursor() as cur:
        for (line_id, attr_code, valor, c, seq) in data:
            cur.execute("""
                INSERT INTO product_attribute_value
                    (external_id, attribute_id, name, html_color, sequence)
                SELECT %s, id, %s, %s, %s
                FROM product_attribute
                WHERE external_id = %s
                ON CONFLICT (attribute_id, name) DO UPDATE SET
                    html_color = EXCLUDED.html_color,
                    write_date = NOW()
            """, (line_id, valor, c, seq, attr_code))
    conn.commit()
    print(f'  ✅ product_attribute_value: {len(data):>5} registros')


def load_templates(conn):
    """Carga product_template."""
    data = [
        (cod, desc, active, precio)
        for cod, (desc, active, precio) in templates.items()
        if desc is not None and str(cod).startswith('A')  # ← solo códigos válidos
    ]
    with conn.cursor() as cur:
        psycopg2.extras.execute_batch(cur, """
            INSERT INTO product_template (default_code, name, active, list_price)
            VALUES (%s, %s, %s, %s)
            ON CONFLICT (default_code) DO UPDATE SET
                name       = EXCLUDED.name,
                active     = EXCLUDED.active,
                list_price = EXCLUDED.list_price,
                write_date = NOW()
        """, data)

        # Templates "fantasma" para variantes huérfanas
        all_template_codes = set(templates.keys())
        ghost_data = []
        for (cod, desc, padre, active, integ, precio) in variants:
            if padre not in all_template_codes:
                ghost_data.append((padre, f'Template {padre}'))
                all_template_codes.add(padre)
        if ghost_data:
            psycopg2.extras.execute_batch(cur, """
                INSERT INTO product_template (default_code, name)
                VALUES (%s, %s)
                ON CONFLICT (default_code) DO NOTHING
            """, ghost_data)

    conn.commit()
    print(f'  ✅ product_template:        {len(data):>5} registros  ({len(ghost_data)} fantasmas)')


def load_variants(conn):
    with conn.cursor() as cur:
        psycopg2.extras.execute_batch(cur, """
            INSERT INTO product_product
                (default_code, product_tmpl_id, name, active, integrable, list_price)
            SELECT %s, pt.id, %s, %s, %s, %s
            FROM product_template pt
            WHERE pt.default_code = %s
            ON CONFLICT (default_code) DO UPDATE SET
                name       = EXCLUDED.name,
                active     = EXCLUDED.active,
                integrable = EXCLUDED.integrable,
                list_price = EXCLUDED.list_price,
                write_date = NOW()
        """, [(cod, desc, active, integ, precio, str(padre))  # ← str(padre)
              for (cod, desc, padre, active, integ, precio) in variants
              if cod and str(cod).startswith('A')])           # ← filtrar códigos válidos
    conn.commit()
    print(f'  ✅ product_product:         {len(variants):>5}')


def load_attribute_lines(conn):
    """Carga product_template_attribute_line."""
    seen = set()
    data = []
    for row in prod_attr_lines:
        tmpl_ref = variant_to_tmpl.get(row['product_ref'])
        if not tmpl_ref:
            continue
        if not isinstance(row['attr_id'], int):
            continue
        key = (tmpl_ref, row['attr_id'])
        if key in seen:
            continue
        seen.add(key)
        data.append((tmpl_ref, row['attr_id']))

    with conn.cursor() as cur:
        psycopg2.extras.execute_batch(cur, """
            INSERT INTO product_template_attribute_line (product_tmpl_id, attribute_id)
            SELECT pt.id, pa.id
            FROM product_template pt, product_attribute pa
            WHERE pt.default_code = %s AND pa.external_id = %s
            ON CONFLICT (product_tmpl_id, attribute_id) DO NOTHING
        """, data)
    conn.commit()
    print(f'  ✅ tmpl_attribute_line:     {len(data):>5} registros')


def load_attribute_values_per_variant(conn):
    """Carga product_template_attribute_value."""
    data = []
    for row in prod_attr_lines:
        if not isinstance(row['attr_id'], int):
            continue
        if not isinstance(row['value_id'], int):
            continue
        tmpl_ref = variant_to_tmpl.get(row['product_ref'])
        if not tmpl_ref:
            continue
        data.append((row['product_ref'], tmpl_ref, row['attr_id'], row['value_id']))

    with conn.cursor() as cur:
        psycopg2.extras.execute_batch(cur, """
            INSERT INTO product_template_attribute_value
                (product_product_id, product_template_attribute_line_id, product_attribute_value_id)
            SELECT pp.id, ptal.id, pav.id
            FROM product_product pp
            JOIN product_template pt   ON pt.id  = pp.product_tmpl_id
            JOIN product_template_attribute_line  ptal ON ptal.product_tmpl_id = pt.id
            JOIN product_attribute                pa   ON pa.id  = ptal.attribute_id
            JOIN product_attribute_value          pav  ON pav.attribute_id = pa.id
            WHERE pp.default_code   = %s
              AND pt.default_code   = %s
              AND pa.external_id    = %s
              AND pav.external_id   = %s
            ON CONFLICT (product_product_id, product_attribute_value_id) DO NOTHING
        """, data, page_size=500)
    conn.commit()
    print(f'  ✅ tmpl_attribute_value:    {len(data):>5} registros')

print('✅ Funciones de carga definidas')

✅ Funciones de carga definidas


In [37]:
for cod, (desc, active, precio) in templates.items():
    if desc is None:
        print(f'Template sin nombre: {cod}')

Template sin nombre: 2299


---
## 6. Ejecutar la carga completa

Ejecuta esta celda cada vez que quieras sincronizar el Excel con la BD.

In [40]:
import time

print('🚀 Iniciando carga...')
t0 = time.time()

conn = get_conn()
try:
    print('\n[1/6] Atributos...')
    load_attributes(conn)

    print('[2/6] Valores de atributo...')
    load_attribute_values(conn)

    print('[3/6] Templates de producto...')
    load_templates(conn)

    print('[4/6] Variantes de producto...')
    load_variants(conn)

    print('[5/6] Líneas de atributo por template...')
    load_attribute_lines(conn)

    print('[6/6] Valores de atributo por variante...')
    load_attribute_values_per_variant(conn)

    elapsed = time.time() - t0
    print(f'\n✅ Carga completada en {elapsed:.1f}s')

except Exception as e:
    conn.rollback()
    print(f'❌ Error durante la carga: {e}')
    raise
finally:
    conn.close()

🚀 Iniciando carga...

[1/6] Atributos...
  ✅ product_attribute:          38 registros
[2/6] Valores de atributo...
  ✅ product_attribute_value:   745 registros
[3/6] Templates de producto...
  ✅ product_template:         1181 registros  (9 fantasmas)
[4/6] Variantes de producto...
  ✅ product_product:          2300
[5/6] Líneas de atributo por template...
  ✅ tmpl_attribute_line:       272 registros
[6/6] Valores de atributo por variante...
  ✅ tmpl_attribute_value:     3893 registros

✅ Carga completada en 1.4s


---
## 7. Verificar resultados

In [ ]:
import pandas as pd

conn = get_conn()
try:
    queries = {
        'product_attribute':       'SELECT COUNT(*) FROM product_attribute',
        'product_attribute_value': 'SELECT COUNT(*) FROM product_attribute_value',
        'product_template':        'SELECT COUNT(*) FROM product_template',
        'product_product':         'SELECT COUNT(*) FROM product_product',
        'tmpl_attribute_line':     'SELECT COUNT(*) FROM product_template_attribute_line',
        'tmpl_attribute_value':    'SELECT COUNT(*) FROM product_template_attribute_value',
    }

    print('📊 Recuento de registros:')
    for tabla, sql in queries.items():
        cur = conn.cursor()
        cur.execute(sql)
        count = cur.fetchone()[0]
        print(f'  {tabla:<30} {count:>6}')

    print('\n📋 Vista v_product_summary (top 10):')
    df = pd.read_sql('SELECT * FROM v_product_summary LIMIT 10', conn)
    display(df)

    print('\n🔍 Vista v_product_variants (primeras 10 líneas):')
    df2 = pd.read_sql('SELECT * FROM v_product_variants LIMIT 10', conn)
    display(df2)

finally:
    conn.close()

---
## 8. (Opcional) Resetear datos sin borrar tablas

⚠️ Usa esto solo si quieres limpiar todos los datos y volver a cargar desde cero.

In [ ]:
# ⚠️  DESCOMENTA SOLO SI QUIERES BORRAR TODOS LOS DATOS

# conn = get_conn()
# with conn.cursor() as cur:
#     cur.execute('TRUNCATE product_template_attribute_value RESTART IDENTITY CASCADE;')
#     cur.execute('TRUNCATE product_template_attribute_line  RESTART IDENTITY CASCADE;')
#     cur.execute('TRUNCATE product_product                  RESTART IDENTITY CASCADE;')
#     cur.execute('TRUNCATE product_template                 RESTART IDENTITY CASCADE;')
#     cur.execute('TRUNCATE product_attribute_value          RESTART IDENTITY CASCADE;')
#     cur.execute('TRUNCATE product_attribute                RESTART IDENTITY CASCADE;')
# conn.commit()
# conn.close()
# print('🗑️  Datos borrados. Puedes volver a ejecutar la celda de carga.')